# Module 7 | Class 4 -- Transformers and HuggingFace

**Objective:** Use a pre-trained transformer model for sentiment analysis and compare its predictions against the TF-IDF + LogReg model from Class 3.

Key ideas:
- Transformers understand context, word order, and negation far better than bag-of-words models
- HuggingFace `pipeline()` gives you production-quality models in 2 lines of code
- No model is perfect -- we will find cases where they disagree

## 1. Setup

In [ ]:
!pip install -q transformers torch scikit-learn nltk pandas

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from transformers import pipeline
import pandas as pd
import numpy as np

print("Setup complete.")

## 2. Load HuggingFace Sentiment Pipeline

The default sentiment-analysis pipeline uses `distilbert-base-uncased-finetuned-sst-2-english`, which was fine-tuned on the Stanford Sentiment Treebank.

In [ ]:
hf_sentiment = pipeline("sentiment-analysis")

# Quick sanity check
result = hf_sentiment("I love this product!")
print(result)

## 3. Run on Sample Reviews

We will test on 10 reviews that range from clearly positive to clearly negative, with a few tricky cases in between.

In [ ]:
reviews = [
    "This product is amazing, best purchase I've ever made!",
    "Terrible quality, broke after one day. Total waste of money.",
    "It's okay, nothing special but it does the job.",
    "Absolutely love it, exceeded all my expectations.",
    "DO NOT buy this, worst experience ever.",
    "The packaging was nice but the product itself is mediocre.",
    "I was skeptical at first, but it actually works really well.",
    "Not great, not terrible. Just average.",
    "Waste of money. Returning immediately.",
    "Surprisingly good for the price, would buy again."
]

hf_results = hf_sentiment(reviews)

print(f"{'Review':<65s} {'Label':>10s} {'Score':>7s}")
print("=" * 85)
for review, result in zip(reviews, hf_results):
    short = review[:60] + "..." if len(review) > 60 else review
    print(f"{short:<65s} {result['label']:>10s} {result['score']:>7.4f}")

## 4. Build a Simple sklearn Model for Comparison

We will train a quick TF-IDF + LogisticRegression model (same approach as Class 3) so we can compare predictions head-to-head.

In [ ]:
import re
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load the same data used in Class 3
try:
    url = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/amazon.csv"
    df = pd.read_csv(url)
    df = df.rename(columns={'reviewText': 'text', 'Positive': 'label'}).dropna(subset=['text', 'label'])
    print(f"Loaded Amazon reviews: {len(df)} rows")
except Exception as e:
    print(f"Download failed ({e}), using 20newsgroups fallback.")
    from sklearn.datasets import fetch_20newsgroups
    cats = ['rec.sport.baseball', 'sci.space']
    raw = fetch_20newsgroups(subset='all', categories=cats, remove=('headers', 'footers', 'quotes'))
    df = pd.DataFrame({'text': raw.data, 'label': raw.target})
    df = df[df['text'].str.strip().str.len() > 20]
    print(f"Loaded 20newsgroups: {len(df)} rows")

stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor', 'never'}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)

df['clean'] = df['text'].apply(clean_text)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

sklearn_model = LogisticRegression(max_iter=1000, random_state=42)
sklearn_model.fit(X_train_tfidf, y_train)

y_pred = sklearn_model.predict(X_test_tfidf)
print(f"sklearn model accuracy on test set: {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
def sklearn_predict(text):
    """Predict sentiment using the sklearn pipeline."""
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    pred = sklearn_model.predict(vec)[0]
    prob = sklearn_model.predict_proba(vec)[0]
    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    return label, max(prob)

print("sklearn predictor ready.")

## 5. Head-to-Head Comparison

In [ ]:
comparison = []

for review, hf_res in zip(reviews, hf_results):
    sk_label, sk_conf = sklearn_predict(review)
    hf_label = hf_res['label']
    hf_conf = hf_res['score']
    agree = "AGREE" if sk_label == hf_label else "DISAGREE"

    comparison.append({
        'review': review[:55],
        'sklearn': f"{sk_label} ({sk_conf:.2f})",
        'huggingface': f"{hf_label} ({hf_conf:.2f})",
        'match': agree
    })

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

In [ ]:
# Highlight disagreements
disagreements = comp_df[comp_df['match'] == 'DISAGREE']
print(f"\nTotal disagreements: {len(disagreements)} out of {len(reviews)}")
if len(disagreements) > 0:
    print("\nDisagreement cases:")
    print(disagreements.to_string(index=False))
else:
    print("Both models agree on all samples.")

## 6. Why Do Models Disagree?

Common reasons:
- **Negation handling:** TF-IDF treats "not good" as two separate words. Transformers understand the phrase.
- **Context:** "skeptical at first, but actually works well" -- the sklearn model sees "skeptical" (negative word) and may be confused. Transformers read the full sentence.
- **Training data:** The HuggingFace model was trained on a different dataset (SST-2) than our sklearn model (Amazon reviews). Different training distributions lead to different decision boundaries.
- **Confidence:** Even when they agree on the label, confidence levels often differ significantly.

---
## TODO: Student Exercise

**Task 1:** Write 5 of your own review sentences. Include:
- At least 1 with negation ("not", "never", etc.)
- At least 1 with sarcasm or mixed sentiment
- At least 1 that is clearly positive and 1 clearly negative

Run both models on them and record results.

**Task 2:** For any disagreement cases (either from our examples or your own),
write a short explanation (2-3 sentences) of why the models likely disagreed.
Consider: negation, context, word-level vs sentence-level understanding.

**Task 3:** Which model would you trust more for a production customer support system? Why?

In [ ]:
# TODO: Task 1 -- Your 5 review sentences
my_reviews = [
    # "Your review here",
]

# Your code here: run both models and display results


In [ ]:
# TODO: Task 2 -- Explain disagreement cases
# Write your explanations as comments or print statements


In [ ]:
# TODO: Task 3 -- Which model for production? (2-3 sentences)
